### dependencies & globals


In [1]:
import requests
import pandas as pd
import geopandas as gpd
from datetime import datetime
import geojson

county_selection = [
    "Fulton",
    "Cobb",
    "DeKalb",
    "Fayette",
    "Gwinnett",
    "Cherokee",
    "Douglas",
    "Forsyth",
    "Clayton",
    "Henry",
    "Rockdale"
]

### API call to get GDOT projects


In [2]:
# Define the base URL of the MapServer layer
base_url = "https://rnhp.dot.ga.gov/hosting/rest/services/GDOT_Public_Outreach/Hub_Project_Search/MapServer/2/query"

# Define query parameters
params = {
    # filter by construction status to include both under construction and pre-construction
    "where": "CONSTRUTION_STATUS_DERIVED = 'PRE-CONSTRUCTION'",
    "outFields": "*",  # Get all fields
    "f": "geojson",  # Request data in GeoJSON format
    "returnGeometry": "true",  # Include geometry data
    "resultOffset": 0,  # Start from the first record
    "resultRecordCount": 1000  # Get 1000 records
}

# list to store each batch of GeoDataFrames
gdfs = []

while True:

    # Make the GET request
    response = requests.get(base_url, params=params)

    # Check if the request was successful
    if response.status_code == 200:
        data = response.json()

        batch_gdf = gpd.GeoDataFrame.from_features(data["features"])

        # if no more data is returned, break the loop
        if batch_gdf.empty:
            break

        gdfs.append(batch_gdf)

        print(
            f"Fetched {len(batch_gdf)} records (offset: {params['resultOffset']})")

        # increase offset by the batch size to fetch the next chunk
        params["resultOffset"] += params["resultRecordCount"]

    else:
        print(
            f"❌ Error fetching data: {response.status_code} - {response.text}")
        break

# Filter out empty GeoDataFrames
gdfs = [gdf.dropna(axis=1, how='all') for gdf in gdfs if not gdf.empty]

# Combine all batches into a single GeoDataFrame
if gdfs:
    full_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))

else:
    print("❌ No data retrieved.")

# drop any columns that ONLY contain NaN values
full_gdf = full_gdf.dropna(axis=1, how="all")

# replace all-caps values in the 'CONSTRUTION_STATUS_DERIVED' column
full_gdf['CONSTRUTION_STATUS_DERIVED'] = full_gdf['CONSTRUTION_STATUS_DERIVED'].str.replace(
    'PRE-CONSTRUCTION', 'Pre-Construction')

# if the 'CONTRACT_DESCRIPTION' column is empty, replace it with the 'SHORT_DESCR' column
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].fillna(
    full_gdf['SHORT_DESCR'])

# define a list of sub-strings to keep in all caps; everything else in the 'SHORT_DESCR' column should be converted to title case
keep_all_caps = ["SR ", "CO ", "CR ", "US ", "RD", "SO ",
                 "CS ", "MI ", "SE ", "NE ", "SW ", "NW ",
                 "NS ", "CSX", "CCTV", "-TIA", " - TIA", "LCI", "GRTA"]

full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.title()
for sub_str in keep_all_caps:
    full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
        sub_str.title(), sub_str)

full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Brdg", "Bridge")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Mtn ", "Mountain ")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Ii", "Phase II")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Ph Ii", "Phase II")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Phase IIi", "Phase III")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Ph IIi", "Phase III")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Iv", "Phase IV")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Phase Vi", "Phase VI")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Resf ", "Resurface ")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Rsrf ", "Resurface ")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Shldr ", "Shoulder ")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Recnst", "Reconstruction")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Cnst ", "Construction ")
full_gdf['CONTRACT_DESCRIPTION'] = full_gdf['CONTRACT_DESCRIPTION'].str.replace(
    "Fm ", "From ")


# filter out records that have either 'WPH' (completed) or 'REJ' (rejected) in the 'CONST_STAT_CD' column
full_gdf = full_gdf[~full_gdf['CONST_STAT_CD'].isin(['WPH', 'REJ'])]

# rename columns
full_gdf = full_gdf.rename(columns={
    'LAST_REFRESH_DTTM': 'Last_refresh',
    'PROJECT_COUNTIES': 'Project_counties',
    'PROJ_ID': 'Project_ID',
    'CONTRACT_DESCRIPTION': 'Project_description',
    'CONTRACTOR_NAME': 'Contractor_name',
    'IS_TIA_PROJECT': 'Is_TIA_project',
})

# drop unneeded columns
full_gdf = full_gdf.drop(columns=[
    'OBJECTID',
    'PRIORITY_CD',
    'SOURCE_OF_CONSTRUCTION_DATES',
    'CONTRACT_ID',
    'CONSTRUTION_STATUS_DERIVED_RSN',
    'PAYMENT_PERCENT_COMPLETE',
    'ESRI_OID',
    'SHORT_DESCR',
    'REC_STATUS',
    'LET_RESP_CD',
    'PRIORITY_CD_DESCR',
    'CONST_STAT_CD',
    'CONSTRUCTION_PERCENT_COMPLETE',
    'CONSTRUTION_STATUS_DERIVED',
    'CURR_COMPLETION_DATE',
    'AWARD_DATE'
])

# create URL column
full_gdf['Project_URL'] = 'https://www.dot.ga.gov/applications/geopi/Pages/Dashboard.aspx?ProjectId=' + \
    full_gdf['Project_ID'].astype(str)

# geographic filter: just get those in the metro
full_gdf = full_gdf[full_gdf['Project_counties'].str.contains(
    '|'.join(county_selection),
    na=False)]

# create a new GeoJSON feature collection with a unique integer ID for each feature
features = []
for i, row in full_gdf.iterrows():
    feature = {
        'type': 'Feature',
        'id': i + 1,  # assign a unique integer ID
        'properties': row.drop('geometry').to_dict(),
        'geometry': row['geometry'].__geo_interface__
    }
    features.append(feature)

# create a new GeoJSON feature collection
collection = {
    'type': 'FeatureCollection',
    'features': features
}

# export the feature collection to a GeoJSON file
with open("data/GDOT_export.geojson", 'w') as f:
    geojson.dump(collection, f)

# export the GeoDataFrame to a CSV file
csv_export = full_gdf.drop(columns=['geometry']).to_csv(
    'GDOT_export.csv', index=False)

# export timestamp file to be inserted in <div> on frontend
current_date = datetime.now().strftime("%B %d, %Y")
with open("data/current_date.txt", "w") as f:
    f.write(current_date)

# # export full_gdf to a geojson file inside the 'data' folder
# full_gdf.to_file("data/GDOT_export.geojson", driver="GeoJSON")
# csv_export = full_gdf.drop(columns=['geometry']).to_csv(
#     'GDOT_export.csv', index=False)

# print status
print(f"✅ Successfully exported {len(full_gdf):,} records!")

# Display final GeoDataFrame
full_gdf

Fetched 1000 records (offset: 0)
Fetched 1000 records (offset: 1000)
Fetched 667 records (offset: 2000)
✅ Successfully exported 481 records!


,geometry,Project_ID,Project_counties,Is_TIA_project,Project_description,Last_refresh,Project_URL
3,"LINESTRING (-84.42751 33.83416, -84.42621 33.8...",M005416,Fulton,No,Trees Along Moores Mill RD @ I-75,1741085725000,https://www.dot.ga.gov/applications/geopi/Page...
5,"LINESTRING (-84.31381 33.92935, -84.31343 33.9...",0019209,DeKalb,No,N Shallowford RD From Chamblee Dunwoody RD To ...,1741085725000,https://www.dot.ga.gov/applications/geopi/Page...
6,"LINESTRING (-84.74175 33.75395, -84.74088 33.7...",0019210,Douglas,No,Fairburn Road From CS 578/Hospital Drive To SR...,1741085725000,https://www.dot.ga.gov/applications/geopi/Page...
7,"LINESTRING (-84.30332 34.04772, -84.30302 34.0...",0019211,Fulton,No,North Point Alpha Link Trail - LCI,1741085725000,https://www.dot.ga.gov/applications/geopi/Page...
8,"LINESTRING (-83.99844 34.0917, -83.99766 34.09...",0019212,Gwinnett,No,Buford Trail Ext From S Lee Street To SR 13 - ...,1741085725000,https://www.dot.ga.gov/applications/geopi/Page...
...,...,...,...,...,...,...,...
2640,"LINESTRING (-84.65716 33.86412, -84.65715 33.8...",770266-,Cobb,No,CS 2446/Hopkins RD @ Wild Horse Creek In Powde...,1741085725000,https://www.dot.ga.gov/applications/geopi/Page...
2641,"LINESTRING (-84.61661 33.63477, -84.61587 33.6...",770271-,Fulton,No,CR 485/Demooney Road @ Deep Creek NW Of Fairburn,1741085725000,https://www.dot.ga.gov/applications/geopi/Page...
2643,"LINESTRING (-84.39644 33.71962, -84.39704 33.7...",M006381,"Clayton , Fulton",No,I-75 From Cw Grant Pkwy/Clayton To NS #926230L...,1741085725000,https://www.dot.ga.gov/applications/geopi/Page...
2654,"LINESTRING (-83.99406 34.0511, -83.99447 34.05...",M006181,Gwinnett,No,SR 20 From CS 1227/S Lee Street To CR 105/Laur...,1741085725000,https://www.dot.ga.gov/applications/geopi/Page...


### misc cleaning
